# arms / training — training-signal health, per arm  `[TRAINING]`

**What this family answers.** For each of the four arms (PTO K=0/K=5, GRPO K=0/K=5, all on one axis):
did the optimiser get a usable signal, and is that signal a faithful proxy for the full-conversation
eval? TensorBoard curves, per-candidate reward spread, method-native advantage/decisiveness, the
degeneration health gate — then the **reward-faithfulness** question: how well does the short
partial-conversation training reward rank conversations like the full-conv eval does, at the length it
actually scores? Exports → `results/arms/training/{figures,tables}/<judge>/`. For a reader debugging
*why* an arm did or didn't learn, or questioning the reward design.

> **Judge handling.** §1–§3 and §6 are pure `[TRAINING]` — produced by the training oracle
> (gpt-4o-mini) during the run and impossible to re-grade after the fact — so they are saved ONLY
> under the primary leaf (`gpt-4o-mini/`); under a held-out `EDA_JUDGE` those sections print a pointer
> instead of emitting byte-identical copies that would imply a measurement that never happened.
> §4–§5 join the training reward to the **eval** side (`S.SCORES`), which *is* grader-dependent, so
> they render once per judge (the held-out leaf = training reward vs the held-out grader's eval).
> Whether the eval instrument itself is trustworthy lives in `measurement/validity`; the **K=0 vs K=5
> faithfulness contrast under a matched policy** lives in `lookahead/mechanism` — not here.


In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)

import os, eda_analysis
from eda_analysis import exports, plotting, stats, training
from eda_analysis.constants import judge_dirname
cfg = eda_analysis.EdaConfig(family="arms/training", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # re-stamp the banner reset_results just removed (it lives under figures/<judge>/)

# Training-side sections (§1-§3, §6) read generations.jsonl / pairs.csv / TB event files — produced by
# the TRAINING oracle and not judge-swappable. They export only under the primary leaf.
TRAINING_SIDE = (S.JUDGE == "")
GRADER = judge_dirname(S.JUDGE)          # short label of the grader behind S.SCORES
PRIMARY_LEAF = f"results/{S.FAMILY}/{{figures,tables}}/{judge_dirname('')}/"
CENSOR = "GRPO K=5 stopped after iteration 5 (censored; iteration_6/ is a one-step stub)."
NUM = {}                                              # number ledger, filled per section, saved at the end
def _pointer(section):
    print(f"[{section}] TRAINING-side section — judge-invariant; saved under the primary leaf only "
          f"({PRIMARY_LEAF}). Skipped for EDA_JUDGE={S.JUDGE!r} ({GRADER}).")
print("TRAINING_SIDE =", TRAINING_SIDE, "| GRADER =", GRADER)

## 1 · Training curves (TensorBoard)  `[TRAINING]`
**Purpose.** The optimiser-side curves per arm, chained across iterations (dotted lines = iteration
boundaries): GRPO loss / reward / reward_std / KL / entropy / completion length; DPO loss / rewards /
margins / accuracies / logps. Parsed straight from the run's event files. One figure per arm →
`figures/<judge>/tb_curves/<arm>.png`. GRPO K=5 was stopped ~2 min into iteration 6, so its curve
carries a one-step `iteration_6` stub after the last dotted line — read it as censored at iteration 5.

In [ ]:
if TRAINING_SIDE:
    for arm in S.ARMS:
        fig = training.tb_curves(arm)
        if fig is None:
            continue
        exports.save_fig(fig, arm.label, group="tb_curves",
                         caption=f"{eda_analysis.arm_label(arm.label)}: TensorBoard training curves parsed from the run's "
                                 f"event files, chained across iterations (dotted vlines = iteration boundaries). "
                                 f"{'GRPO' if arm.method == 'GRPO' else 'DPO'} scalars as logged by TRL "
                                 f"(training oracle gpt-4o-mini; not judge-swappable). "
                                 + (CENSOR if arm.label == 'GRPO_LA5' else ''))
        plt.show()
else:
    _pointer("§1 TB curves")

## 2 · Candidate reward + advantage signal  `[TRAINING]`
**Purpose.** What the oracle handed back per candidate (partial-branch reward), and how **decisively**
it separates a branch's candidates — on ONE comparable *oracle-score-gap* scale. The like-for-like
signal (solid, **both** methods) is the **unfiltered best−worst candidate range**; each method's native
secondary is faint: GRPO `group_std`, PTO the **τ-filtered chosen−rejected margin** (the actual DPO
signal). **Read:** the per-branch spreads are modest and comparable (~0.2–0.3 oracle points), GRPO's
*marginally* wider than PTO's on the honest unfiltered basis. PTO's τ-filtered margin sits slightly
**above** its own unfiltered range because τ keeps only large-gap branches — so a margin-vs-range
comparison overstates PTO's decisiveness. In *shape*, PTO's range declines steadily while GRPO's dips
mid-training then **rebounds late** (the K=0 iter-8 reward-hack echo). Look-ahead widens the spread
(K=5 > K=0 on both range and margin — descriptive, per arm; the tested K contrast is in
`lookahead/mechanism`). Combined `reward_distribution_by_arm.png` + a per-arm zoom in
`reward_distribution/<arm>.png`; the numbers behind both figures are the tables
`reward_distribution_by_iter` and `advantage_signal_by_iter`.

⚠ PTO's `branch_id` is the trunk DEPTH, not a unique id — every per-branch aggregate here keys on
`(conversation_id, branch_id)` (see `training.advantage_signal_by_iter`).

In [ ]:
if TRAINING_SIDE:
    RWD = training.reward_distribution_frame(S.ARMS)
    fig = plotting.reward_distribution(RWD)
    if fig:
        exports.save_fig(fig, "reward_distribution_by_arm",
                         caption="Per-candidate training reward per training iteration, one panel per arm (all four arms; "
                                 "training oracle gpt-4o-mini on partial-conv branches, NOT the full-conv eval; unit = "
                                 "one scored candidate). " + CENSOR)
        plt.show()
    # Per-arm zoom — the reward_distribution/ subfolder companion to the combined grid above.
    if not RWD.empty:
        for arm in sorted(RWD.arm.unique()):
            figa = plotting.reward_distribution(RWD[RWD.arm == arm], ncols=1)
            if figa is None:
                continue
            exports.save_fig(figa, arm, group="reward_distribution",
                             caption=f"{eda_analysis.arm_label(arm)}: per-candidate training-reward distribution per "
                                     f"training iteration (partial-conv training oracle gpt-4o-mini, not the full-conv eval).")
            plt.show()
        RWD_SUM = (RWD.groupby(["arm", "method", "train_iter"], observed=True)["score"]
                   .agg(n_candidates="size", mean="mean", sd="std", q10=lambda s: s.quantile(.10),
                        median="median", q90=lambda s: s.quantile(.90), pct_floor=lambda s: 100 * (s <= 0).mean())
                   .reset_index())
        exports.save_table(RWD_SUM, "reward_distribution_by_iter",
                           caption="Per (arm, training iteration): candidate-count, mean/sd/deciles of the per-candidate "
                                   "training reward, and % of candidates at the reward floor (training oracle gpt-4o-mini "
                                   "on partial-conv branches; the numbers behind reward_distribution_by_arm). " + CENSOR)
        display(RWD_SUM.round(3))

    ADV = training.advantage_signal_by_iter(S.ARMS)
    fig = plotting.advantage_signal_sidebyside(ADV)
    if fig:
        from matplotlib.ticker import MaxNLocator
        for _ax in fig.axes:                       # training iteration is an integer axis (GRPO K=5 has only 5)
            _ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        exports.save_fig(fig, "advantage_signal_sidebyside",
                         caption="Training decisiveness on ONE comparable oracle-score-gap scale, all four arms: the UNFILTERED "
                                 "best-worst candidate range (solid, both methods) is the like-for-like signal; GRPO "
                                 "within-group std / PTO tau-filtered chosen-rejected margin are faint secondaries. "
                                 "Per-branch spreads are modest and comparable (~0.2-0.3), GRPO marginally wider; PTO's "
                                 "tau-filtered margin sits slightly above its own range (tau keeps only large-gap branches). "
                                 "PTO's range declines steadily; GRPO's dips then rebounds late (K=0 iter-8 hack echo). "
                                 "K=5 > K=0 on both (descriptive; per-branch unit keyed on (conversation_id, branch_id)). " + CENSOR)
        plt.show()
    if not ADV.empty:
        exports.save_table(ADV, "advantage_signal_by_iter",
                           caption="Per (arm, training iteration) decisiveness signal: group_range = mean unfiltered best-worst "
                                   "candidate reward per branch (both methods, like-for-like); GRPO group_std / frac_zero_std "
                                   "(within-group spread, collapsed-group share); PTO margin / margin_median / n_pairs from the "
                                   "tau-filtered pairs.csv (the DPO signal). Training oracle gpt-4o-mini. " + CENSOR)
        display(ADV.round(3))
        for arm, g in ADV.groupby("arm"):
            NUM[f"advantage.{arm}.mean_group_range"] = {"value": float(g.group_range.mean()),
                                                        "source": "tables/advantage_signal_by_iter.md",
                                                        "note": "mean over training iterations of the unfiltered best-worst candidate range"}
            if g.margin.notna().any():
                NUM[f"advantage.{arm}.mean_margin"] = {"value": float(g.margin.mean()),
                                                       "source": "tables/advantage_signal_by_iter.md",
                                                       "note": "mean over training iterations of the tau-filtered chosen-rejected margin"}
            if g.group_std.notna().any():
                NUM[f"advantage.{arm}.mean_group_std"] = {"value": float(g.group_std.mean()),
                                                          "source": "tables/advantage_signal_by_iter.md",
                                                          "note": "mean over training iterations of GRPO within-group reward std"}
else:
    _pointer("§2 candidate reward + advantage")

## 3 · Degeneration check  `[TRAINING]`
**Purpose.** Confirm the ChatML-leak / empty / floored-completion fixes held in the real runs.
**Read:** `pct_leak`/`pct_empty` near 0 = clean; a spike flags a generation pathology that turn. This
is the health gate, not a result — table `degeneration_scan`.

In [ ]:
if TRAINING_SIDE:
    DEG = training.scan_degeneracy(training.load_generations(S.ARMS))
    if not DEG.empty:
        DEG_T = DEG[["arm", "train_iter", "n_candidates", "pct_leak", "pct_empty", "pct_floored", "mean_score", "mean_len"]]
        exports.save_table(DEG_T, "degeneration_scan",
                           caption="Per (arm, training iteration): candidate count and degeneration rates -- % ChatML-marker "
                                   "leaks, % empty-after-clean, % floored to REWARD_FLOOR -- plus mean training reward and mean "
                                   "completion length (chars). Near-zero = the 2026-06-07 stop-string fixes held. Training "
                                   "oracle gpt-4o-mini. " + CENSOR)
        display(DEG_T)
        for arm, g in DEG.groupby("arm"):
            NUM[f"degeneration.{arm}.max_pct_leak"] = {"value": float(g.pct_leak.max()), "source": "tables/degeneration_scan.md",
                                                       "note": "worst training iteration"}
            NUM[f"degeneration.{arm}.max_pct_empty"] = {"value": float(g.pct_empty.max()), "source": "tables/degeneration_scan.md",
                                                        "note": "worst training iteration"}
            NUM[f"degeneration.{arm}.n_candidates_total"] = {"value": int(g.n_candidates.sum()), "source": "tables/degeneration_scan.md",
                                                             "note": "scored candidates across training iterations"}
    else:
        print("No generations.jsonl rows on disk for these arms.")
else:
    _pointer("§3 degeneration")

## 4 · Reward faithfulness — rank agreement vs conversation length  `[TRAINING ↔ EVAL]`

**Purpose.** Rebuild the Exp2 partial-conv statistic on Exp3 data (from the per-branch `prefix` in
`generations.jsonl`, no new oracle calls): for each scored length `n_turns`, the fraction of
conversation pairs whose proxy-score ordering matches the full-conv eval ordering (pairs formed within
(arm, eval_iter, n_turns), pooled per (arm, n_turns)). **Read:** 0.5 = chance; higher = more faithful.
Per arm, all four on one axis: LA5 above LA0 *suggests* look-ahead makes the short reward more
faithful — the descriptive read; the tested contrast (matched policy, both graders, bootstrap CIs) is
`lookahead/mechanism`. MCL=12 keeps the shortest training cut out of the unreliable regime (Exp2 saw
0.66 at `n_turns=2`).

The proxy is always the **training** oracle (gpt-4o-mini, partial branch); the eval side is the grader
this leaf is rendered under (`S.SCORES`) — so the held-out leaf reads "training reward vs the held-out
grader's full-conv eval". `n_pairs` counts all pairwise conv comparisons and overstates independent
information: read the agreement as a descriptive fraction, not a CI-bearing estimate.

In [ ]:
BR = training.load_branch_reliability(S.ARMS)
RA = stats.rank_agreement_by_nturns(BR, S.SCORES, metric="Q1Q2")
fig = plotting.reliability_curve(RA, palette=S.PALETTE)
if fig:
    fig.axes[0].set_ylabel(f"sign-agreement with full-conv eval ({GRADER})")
    exports.save_fig(fig, "reward_reliability_curve",
                     caption=f"Sign-agreement between the partial-conv TRAINING reward (training oracle gpt-4o-mini, chosen "
                             f"branch) and the full-conv EVAL Q1+Q2 as graded by {GRADER}, vs the length at which the "
                             f"conversation was scored (n_turns); one line per arm, all four arms. 0.5 = chance. Pairs of "
                             f"conversations within (arm, eval_iter, n_turns), pooled per (arm, n_turns); descriptive "
                             f"(LA5 above LA0 suggests look-ahead improves faithfulness -- the tested contrast is "
                             f"lookahead/mechanism). {CENSOR}")
    plt.show()
if not RA.empty:
    RA_PIV = RA.pivot_table(index="n_turns", columns="arm", values="agreement").round(4)
    exports.save_table(RA_PIV.reset_index(), "reward_reliability_by_nturns",
                       caption=f"Rank agreement (fraction of conversation pairs whose training-proxy ordering matches the "
                               f"full-conv eval ordering, eval graded by {GRADER}) per scored length n_turns, one column per "
                               f"arm; 0.5 = chance. Long form with n_pairs in reward_reliability_by_nturns_long. {CENSOR}")
    exports.save_table(RA.sort_values(["arm", "n_turns"]).reset_index(drop=True), "reward_reliability_by_nturns_long",
                       float_format="%.4f",
                       caption=f"Long form of reward_reliability_by_nturns: (arm, n_turns, agreement, n_pairs); eval graded by "
                               f"{GRADER}; n_pairs counts every within-(arm, eval_iter, n_turns) conversation pair, so it "
                               f"overstates independent information. {CENSOR}")
    display(RA_PIV)
    for arm, g in RA.groupby("arm"):
        g = g.sort_values("n_turns")
        NUM[f"faithfulness.{GRADER}.{arm}.agreement_at_min_nturns"] = {
            "value": float(g.agreement.iloc[0]), "source": "tables/reward_reliability_by_nturns_long.md",
            "note": f"n_turns={int(g.n_turns.iloc[0])} (the shortest scored cut, = MCL), n_pairs={int(g.n_pairs.iloc[0])}"}
        NUM[f"faithfulness.{GRADER}.{arm}.agreement_at_max_nturns"] = {
            "value": float(g.agreement.iloc[-1]), "source": "tables/reward_reliability_by_nturns_long.md",
            "note": f"n_turns={int(g.n_turns.iloc[-1])} (the longest bin with >= 20 pairs), n_pairs={int(g.n_pairs.iloc[-1])}"}

## 5 · Proxy reward vs full-conv eval, per iteration  `[TRAINING ↔ EVAL]`
**Purpose.** The aggregate view: mean training proxy reward vs mean full-conv eval per (arm, iteration),
dashed y=x (join `eval_iter = train_iter − 1`: training iteration N branches the policy that produced
the `model_iter_{N−1}` eval convs). **Read:** points below the line = the proxy over-rates the policy
(promising openings that don't pay off by session end). Eval side = the grader of this leaf.

In [ ]:
GENS = training.load_generations(S.ARMS)
fig = plotting.faithfulness_proxy_vs_eval(S.SCORES, GENS, palette=S.PALETTE)
if fig:
    fig.axes[0].set_ylabel(f"eval Q1Q2 ({GRADER})")
    fig.axes[0].set_xlabel("proxy (mean training reward, partial branch; K=5 arms: look-ahead-scored)")
    exports.save_fig(fig, "faithfulness_proxy_vs_eval",
                     caption=f"Per (arm, iteration), all four arms: mean TRAINING proxy reward (training oracle gpt-4o-mini on "
                             f"the partial branch, every candidate) vs mean full-conversation EVAL Q1+Q2 graded by {GRADER}; "
                             f"dashed y=x; label = eval iteration (= training iteration - 1). Points below the line = the "
                             f"proxy over-rates the policy. {CENSOR}")
    plt.show()
if not GENS.empty:
    _proxy = (GENS.groupby(["arm", "eval_iter"])["score"].mean().rename("proxy").reset_index()
              .rename(columns={"eval_iter": "iteration"}))
    _proxy_n = (GENS.groupby(["arm", "eval_iter"])["score"].size().rename("n_candidates").reset_index()
                .rename(columns={"eval_iter": "iteration"}))
    _eval = (S.SCORES[S.SCORES.questionnaire == "Q1Q2"].groupby(["arm", "iteration"])["score"]
             .agg(eval="mean", n_convs="size").reset_index())
    FAITH = _proxy.merge(_proxy_n, on=["arm", "iteration"]).merge(_eval, on=["arm", "iteration"])
    FAITH["proxy_minus_eval"] = FAITH["proxy"] - FAITH["eval"]
    FAITH = FAITH.sort_values(["arm", "iteration"]).reset_index(drop=True)
    exports.save_table(FAITH, "faithfulness_proxy_vs_eval",
                       caption=f"Per (arm, eval iteration): mean training proxy reward (all candidates of training iteration "
                               f"N = eval iteration N-1; training oracle gpt-4o-mini) vs mean full-conv eval Q1+Q2 graded by "
                               f"{GRADER}, and their difference (> 0 = the proxy over-rates the policy). {CENSOR}")
    display(FAITH.round(3))
    for arm, g in FAITH.groupby("arm"):
        NUM[f"proxy_vs_eval.{GRADER}.{arm}.mean_proxy_minus_eval"] = {
            "value": float(g.proxy_minus_eval.mean()), "source": "tables/faithfulness_proxy_vs_eval.md",
            "note": "mean over iterations of (training proxy - full-conv eval Q1Q2); > 0 = proxy over-rates"}

## 6 · PTO preference decisiveness by branch depth  `[TRAINING]`
**Purpose.** Within PTO, how the chosen−rejected score `margin` varies with `branch_depth` (deeper =
later in the trunk = longer prefix). **Read:** a margin that grows with depth = the oracle
discriminates more confidently on longer context. PTO-only — GRPO has no pairs. Figure + table
`pto_margin_by_depth` (was inline-only before the reorg).

In [ ]:
if TRAINING_SIDE:
    PP = training.load_pref_pairs([a for a in S.ARMS if a.method == "PTO"])
    if not PP.empty and "branch_depth" in PP.columns:
        MBD = (PP.groupby(["arm", "branch_depth"])["margin"]
               .agg(n_pairs="size", margin_mean="mean", margin_median="median", margin_sd="std").reset_index())
        fig, ax = plt.subplots(figsize=(8, 4.2))
        sns.lineplot(MBD, x="branch_depth", y="margin_mean", hue="arm", marker="o",
                     palette=plotting.arm_palette(sorted(MBD.arm.unique())), ax=ax)
        ax.axhline(0, color="grey", lw=0.6, ls="--"); ax.set_title("PTO chosen-rejected margin by branch depth")
        ax.set_xlabel("branch depth (deeper = longer prefix)"); ax.set_ylabel("mean score margin (chosen - rejected)")
        plotting.relabel_legend(ax); fig.tight_layout()
        exports.save_fig(fig, "pto_margin_by_depth",
                         caption="PTO only (both K arms): mean chosen-rejected training-reward margin of the tau-filtered "
                                 "preference pairs by branch depth in the greedy trunk (deeper = longer prefix); training "
                                 "oracle gpt-4o-mini; unit = one emitted pair, pooled over training iterations. A margin "
                                 "growing with depth = the oracle discriminates more confidently on longer context.")
        plt.show()
        exports.save_table(MBD, "pto_margin_by_depth",
                           caption="PTO only: per (arm, branch_depth) count of tau-filtered pairs and mean / median / sd of the "
                                   "chosen-rejected training-reward margin, pooled over training iterations (training oracle "
                                   "gpt-4o-mini).")
        display(MBD.round(3))
    else:
        print("No PTO preference pairs available.")
else:
    _pointer("§6 PTO margin by depth")

## 7 · How to read this family
- **Curves (§1)** are the ground truth of optimisation; a flat/exploding loss or collapsing `reward_std`
  explains a stalled arm. GRPO K=5 is censored at iteration 5 everywhere in this family.
- **Advantage (§2)** is shown as the comparable oracle-score gap: PTO's chosen−rejected `margin`
  declines steadily (signal saturating), while GRPO's best−worst `group_range` dips mid-training then
  **rebounds late** (the K=0 iter-8 reward-hack echo) — so it is *not* a clean monotone shrink. (The
  τ-filtered PTO margin sits beside the unfiltered PTO range, plotted so they can be read like-for-like.)
- **Degeneration (§3)** should stay ~0; this is the health gate, not a result.
- The **reliability curve (§4)** is the faithfulness headline: it quantifies how much the training
  reward can be trusted at the length it actually scores. MCL=12 was chosen to stay out of the
  unreliable short-cut regime. Reading LA5 against LA0 here is descriptive; the tested K contrast
  (matched policy, both graders, bootstrap CIs) is `lookahead/mechanism`. *(`n_pairs` counts all
  pairwise conv comparisons, so it overstates independent information — a descriptive fraction, not
  a CI-bearing estimate.)*
- **Is the *grader* trustworthy?** That is a different question and it lives in
  `measurement/validity`: oracle ICC, the decoupled second judge, the multi-judge variance
  decomposition and gain retention. §4 asks whether the *training* reward predicts the *eval*
  score; `measurement/validity` asks whether the eval score itself is reproducible and grader-independent.
- _(Held-out outcomes: `arms/outcomes`; questionnaire detail: `arms/questionnaires`; validity/hacking:
  `arms/validity`; the preference-pair analysis: `arms/preference`.)_

The ledger `tables/<judge>/training_numbers.json` collects the per-arm anchor numbers of this family
(sources point at the tables above).

In [ ]:
if NUM:
    exports.save_numbers("training_numbers", NUM,
                         caption=f"Number ledger for arms/training: per-arm anchors -- decisiveness means (§2), degeneration "
                                 f"maxima (§3), rank agreement at the shortest/longest scored cut (§4) and mean proxy-eval gap "
                                 f"(§5); eval side graded by {GRADER}, training side always the training oracle gpt-4o-mini "
                                 f"(§2/§3 keys present only under the primary leaf). {CENSOR}")
    print(f"ledger: {len(NUM)} keys")

In [ ]:
exports.prune_orphan_captions(); print("index ->", exports.build_index())